In [1]:
import squidpy as sq
import scanpy as sc

from numpy.random import default_rng
from anndata import AnnData
import scipy
from sklearn.metrics.cluster import adjusted_rand_score
import numpy as np
import pandas as pd
import seaborn as sns
import os
from sklearn import metrics
import multiprocessing as mp
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score, homogeneity_score, completeness_score
import pickle
from sklearn.neighbors import kneighbors_graph
from sklearn.mixture import GaussianMixture
from sklearn.mixture import BayesianGaussianMixture

In [2]:
from numpy import genfromtxt
import time

In [3]:
from SpaceFlow import SpaceFlow

c:\Users\ADMIN\anaconda3\envs\divideconquer\lib\site-packages\torch_geometric\typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
c:\Users\ADMIN\anaconda3\envs\divideconquer\lib\site-packages\torch_geometric\typing.py:124: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "


In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import time
from anndata import AnnData
from sklearn.metrics import normalized_mutual_info_score
from numpy import genfromtxt
import SpaceFlow 

folder_path = "E:/SingleCellCluster/Dataset/Visium"

file_list = ["151507.h5ad", "151508.h5ad", "151509.h5ad", "151510.h5ad"]


sf_embedding_all = np.empty((0, 50), dtype=np.float32)
batch_labels = []
batch_index = 0
sf_meta = []
true_label_all = []  
slice_all = [] 



for file in file_list:
    
    slice_name = file.split(".")[0]
    
    print(f"🔄 Đang xử lý: {slice_name}")
    file_path = folder_path + f"/{file}"
    adata = sc.read_h5ad(file_path)
    
   
    if 'X_pca' not in adata.obsm:
        print("ℹ️ Tính PCA cho AnnData...")
        sc.pp.normalize_total(adata)
        sc.pp.log1p(adata)
        sc.pp.pca(adata, n_comps=50)
    
    adata.obsm["spatial"] = adata.obsm["spatial"]
    true_labels = pd.factorize(adata.obs["Region"])[0] 
    true_label_all += list(true_labels)  
    slice_names = adata.obs["Region"].tolist()
    slice_all += slice_names

    
    if adata.shape[0] > 10:
        time_st = time.time()
        data_name = slice_name
        
        adataNew = AnnData(1.0 + adata.obsm['X_pca'] - adata.obsm['X_pca'].min())
        adataNew.obsm['spatial'] = adata.obsm['spatial']
        
        
        from SpaceFlow.SpaceFlow import SpaceFlow  
        sf = SpaceFlow(adata=adataNew)

        sf.preprocessing_data(n_top_genes=40)
        
        embed_filepath = f'output/{data_name}SpaceFlowembedding.tsv'
        domain_filepath = f'output/{data_name}SpaceFlowdomains.tsv'
        
        sf.train(
            embedding_save_filepath=embed_filepath,
            spatial_regularization_strength=0.1,
            z_dim=50,
            lr=1e-3,
            epochs=300,
            max_patience=30,
            min_stop=100,
            random_seed=42,
            gpu=0,
            regularization_acceleration=True,
            edge_subset_sz=1000000
        )
        
        # Load kết quả
        sf_embedding = genfromtxt(embed_filepath, delimiter='\t')
        embedding_adata = AnnData(sf_embedding)
        sc.pp.neighbors(embedding_adata, n_neighbors=50, use_rep='X')
        sc.tl.leiden(embedding_adata, resolution=1.0)
        cluster_SpaceFlow = embedding_adata.obs["leiden"].cat.codes.values
        
        time_ed = time.time()
        time_spaceflow = time_ed - time_st
        
        # Lưu nhãn cluster + ground truth
        df_out = pd.DataFrame({
            "SpaceFlow": cluster_SpaceFlow,
            "ground_truth": true_labels.astype(int)
        })
        out_data = f'outdata/{data_name}SpaceFlow.csv'
        df_out.to_csv(out_data, index=False)
        
        # Gộp embedding và meta
        sf_embedding_all = np.vstack([sf_embedding_all, sf_embedding])
        batch_labels += [batch_index] * sf_embedding.shape[0]
        batch_index += 1
        
        sf_info = [
            data_name,
            sf_embedding.shape[0],
            normalized_mutual_info_score(true_labels, cluster_SpaceFlow),
            time_spaceflow
        ]
        sf_meta.append(sf_info)


🔄 Đang xử lý: 151507
ℹ️ Tính PCA cho AnnData...
Epoch 2/300, Loss: 2.8570716381073
Epoch 12/300, Loss: 1.4988410472869873
Epoch 22/300, Loss: 1.476409673690796
Epoch 32/300, Loss: 1.476660132408142
Epoch 42/300, Loss: 1.4775469303131104
Epoch 52/300, Loss: 1.4785996675491333
Epoch 62/300, Loss: 1.478562355041504
Epoch 72/300, Loss: 1.4768927097320557
Epoch 82/300, Loss: 1.4757404327392578
Epoch 92/300, Loss: 1.4761346578598022
Epoch 102/300, Loss: 1.4742631912231445
Epoch 112/300, Loss: 1.4748146533966064
Epoch 122/300, Loss: 1.4759509563446045
Epoch 132/300, Loss: 1.474760890007019
Epoch 142/300, Loss: 1.47581148147583
Epoch 152/300, Loss: 1.474381446838379
Epoch 162/300, Loss: 1.474199652671814
Epoch 172/300, Loss: 1.4735546112060547
Training complete!
Embedding is saved at output/151507SpaceFlowembedding.tsv


c:\Users\ADMIN\anaconda3\envs\divideconquer\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔄 Đang xử lý: 151508
ℹ️ Tính PCA cho AnnData...
Epoch 2/300, Loss: 2.964401960372925
Epoch 12/300, Loss: 1.5338773727416992
Epoch 22/300, Loss: 1.660239338874817
Epoch 32/300, Loss: 1.533845067024231
Epoch 42/300, Loss: 1.4805036783218384
Epoch 52/300, Loss: 1.4797000885009766
Epoch 62/300, Loss: 1.4799070358276367
Epoch 72/300, Loss: 1.4797645807266235
Epoch 82/300, Loss: 1.4791475534439087
Epoch 92/300, Loss: 1.4779542684555054
Epoch 102/300, Loss: 1.4779226779937744
Epoch 112/300, Loss: 1.477890133857727
Epoch 122/300, Loss: 1.4781689643859863
Epoch 132/300, Loss: 1.4783844947814941
Epoch 142/300, Loss: 1.4777888059616089
Epoch 152/300, Loss: 1.4783259630203247
Epoch 162/300, Loss: 1.4781379699707031
Epoch 172/300, Loss: 1.4765657186508179
Epoch 182/300, Loss: 1.4760305881500244
Epoch 192/300, Loss: 1.4761489629745483
Epoch 202/300, Loss: 1.4753681421279907
Epoch 212/300, Loss: 1.473962664604187
Epoch 222/300, Loss: 1.4728246927261353
Epoch 232/300, Loss: 1.4727599620819092
Epoch 24

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import scanpy.external as sce
from sklearn.metrics import normalized_mutual_info_score, homogeneity_score, completeness_score, adjusted_rand_score
import time


sf_meta_df = pd.DataFrame(
    sf_meta,
    columns=['metaSlice', 'n', 'NMI', 'SpaceFlowRuntime']
)
sf_meta_df.to_csv('sf_meta_df_runtime_round1.csv', index=False)


adata_combine = sc.AnnData(sf_embedding_all)
adata_combine.obsm['X_pca'] = sf_embedding_all
adata_combine.obs['batch'] = pd.Categorical(np.array(batch_labels))
adata_combine.obs['true_label'] = pd.Categorical(true_label_all)  # 👈 THÊM NHÃN GROUND TRUTH TẠI ĐÂY
adata_combine.obs['slice'] = pd.Categorical(slice_all)


time_st = time.time()
sce.pp.harmony_integrate(adata_combine, key='batch')
time_ed = time.time()
time_harmony = time_ed - time_st

time_st = time.time()
sc.pp.neighbors(adata_combine, use_rep='X_pca_harmony')
sc.tl.leiden(adata_combine, resolution=0.5)
time_ed = time.time()
time_leiden = time_ed - time_st

combined_labels = pd.factorize(adata_combine.obs['true_label'])[0]
pred_labels = adata_combine.obs['leiden'].astype(int).values


ari_per_slice = []  

for slice_name in adata_combine.obs['slice'].unique():
    
    slice_idx = adata_combine.obs['slice'] == slice_name
    
   
    true_labels_slice = combined_labels[slice_idx]
    pred_labels_slice = pred_labels[slice_idx]
    

    ari_slice = round(adjusted_rand_score(true_labels_slice, pred_labels_slice), 6)
    ari_per_slice.append(ari_slice)
    
    print(f"📊 ARI cho Slice {slice_name}: {ari_slice}")


valid_idx = combined_labels >= 0


nmi = round(normalized_mutual_info_score(combined_labels[valid_idx], pred_labels[valid_idx]), 6)
homo = round(homogeneity_score(combined_labels[valid_idx], pred_labels[valid_idx]), 6)
comp = round(completeness_score(combined_labels[valid_idx], pred_labels[valid_idx]), 6)


ari = round(adjusted_rand_score(combined_labels[valid_idx], pred_labels[valid_idx]), 6)


sf_meta_df['Harmony_combine_Time'] = [time_harmony] * len(sf_meta_df)
sf_meta_df['time_leiden'] = [time_leiden] * len(sf_meta_df)
sf_meta_df['ARI'] = [ari] * len(sf_meta_df)  
sf_meta_df.to_csv('sf_meta_df_runtime.csv', index=False)


print("✅ Đánh giá chất lượng phân cụm sau khi tích hợp:")
print("  📊 Normalized Mutual Information (NMI):", nmi)
print("  📊 Homogeneity Score:", homo)
print("  📊 Completeness Score:", comp)
print("  📊 Adjusted Rand Index (ARI):", ari)


2025-06-18 16:38:12,306 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2025-06-18 16:38:15,246 - harmonypy - INFO - sklearn.KMeans initialization complete.
2025-06-18 16:38:15,382 - harmonypy - INFO - Iteration 1 of 10
2025-06-18 16:38:18,546 - harmonypy - INFO - Iteration 2 of 10
2025-06-18 16:38:21,618 - harmonypy - INFO - Iteration 3 of 10
2025-06-18 16:38:25,155 - harmonypy - INFO - Converged after 3 iterations


📊 ARI cho Slice 151507: 0.358423
📊 ARI cho Slice 151508: 0.421788
📊 ARI cho Slice 151509: 0.411029
📊 ARI cho Slice 151510: 0.407693
✅ Đánh giá chất lượng phân cụm sau khi tích hợp:
  📊 Normalized Mutual Information (NMI): 0.362713
  📊 Homogeneity Score: 0.468175
  📊 Completeness Score: 0.296029
  📊 Adjusted Rand Index (ARI): 0.187101
